# Fashion Retail Discount Analysis

## Project Overview
This project analyses transactional and discount data from a global fashion retailer
operating across 7 countries and 35 stores over a 2-year period.

## Business Question
Does discounting actually drive more sales — or has constant discounting
trained customers to buy regardless of whether a discount exists?

This is a question I observed firsthand working in retail at Strandbags,
where promotional sales were near-constant. I wanted to test with real data
whether discounts genuinely move the needle on sales volume and revenue.

## Approach
- Python (pandas, numpy) — data loading, cleaning & feature engineering
- SQL (sqlite3) — business questions & aggregations
- Tableau — interactive dashboard

## Data Source
Global Fashion Retail Sales dataset — synthetic transactional data simulating
2 years of sales across 35 stores in 7 countries.
Source: Kaggle (ricgomes/global-fashion-retail-stores-dataset)

## 1. Loading the Data

Loading all 6 files from the dataset — transactions, discounts, products,
stores, employees and customers. I'll only use what's relevant to the
discount analysis, but good to have everything available.

In [2]:
import pandas as pd
import numpy as np
import io
from google.colab import files

uploaded = files.upload()

Saving customers.csv to customers.csv
Saving discounts.csv to discounts.csv
Saving employees.csv to employees.csv
Saving products.csv to products.csv
Saving stores.csv to stores.csv
Saving transactions.csv to transactions.csv


In [3]:
transactions = pd.read_csv(io.BytesIO(uploaded['transactions.csv']))
discounts = pd.read_csv(io.BytesIO(uploaded['discounts.csv']))
products = pd.read_csv(io.BytesIO(uploaded['products.csv']))
stores = pd.read_csv(io.BytesIO(uploaded['stores.csv']))
employees = pd.read_csv(io.BytesIO(uploaded['employees.csv']))
customers = pd.read_csv(io.BytesIO(uploaded['customers.csv']))

print("transactions:", transactions.shape)
print("discounts:", discounts.shape)
print("products:", products.shape)
print("stores:", stores.shape)
print("employees:", employees.shape)
print("customers:", customers.shape)

transactions: (6416827, 19)
discounts: (181, 6)
products: (17940, 12)
stores: (35, 8)
employees: (404, 4)
customers: (1643306, 9)


/tmp/ipykernel_2851/931464052.py:6: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  customers = pd.read_csv(io.BytesIO(uploaded['customers.csv']))


## 2. Initial Exploration

Before cleaning anything, I want to understand what each key file
looks like — column names, data types, and missing values.

In [4]:
print("=== TRANSACTIONS ===")
print(transactions.dtypes)
print("\nMissing values:")
print(transactions.isna().sum())

=== TRANSACTIONS ===
Invoice ID           object
Line                  int64
Customer ID           int64
Product ID            int64
Size                 object
Color                object
Unit Price          float64
Quantity              int64
Date                 object
Discount            float64
Line Total          float64
Store ID              int64
Employee ID           int64
Currency             object
Currency Symbol      object
SKU                  object
Transaction Type     object
Payment Method       object
Invoice Total       float64
dtype: object

Missing values:
Invoice ID                0
Line                      0
Customer ID               0
Product ID                0
Size                 413102
Color               4350783
Unit Price                0
Quantity                  0
Date                      0
Discount                  0
Line Total                0
Store ID                  0
Employee ID               0
Currency                  0
Currency Symbol         

In [5]:
print(transactions.head(3))
print("\nDiscount unique values sample:")
print(transactions['Discount'].value_counts().head(10))
print("\nTransaction Type values:")
print(transactions['Transaction Type'].value_counts())

            Invoice ID  Line  Customer ID  Product ID Size    Color  \
0  INV-US-001-03558761     1        47162         485    M      NaN   
1  INV-US-001-03558761     2        47162        2779    G      NaN   
2  INV-US-001-03558761     3        47162          64    M  NEUTRAL   

   Unit Price  Quantity                 Date  Discount  Line Total  Store ID  \
0        80.5         1  2023-01-01 15:42:00       0.0        80.5         1   
1        31.5         1  2023-01-01 15:42:00       0.4        18.9         1   
2        45.5         1  2023-01-01 15:42:00       0.4        27.3         1   

   Employee ID Currency Currency Symbol               SKU Transaction Type  \
0            7      USD               $        MASU485-M-             Sale   
1            7      USD               $       CHCO2779-G-             Sale   
2            7      USD               $  MACO64-M-NEUTRAL             Sale   

  Payment Method  Invoice Total  
0           Cash          126.7  
1           C

## 3. Data Cleaning

Key issues to fix:
- Filter out returns, keep only Sales
- Fix Date column from text to datetime
- Drop Color column (68% missing, not needed for analysis)
- Rename 'Discont' column in discounts file (typo)
- Standardise column names (remove spaces)

In [6]:
transactions = transactions[transactions['Transaction Type'] == 'Sale']
print("Rows after removing returns:", transactions.shape[0])

Rows after removing returns: 6077200


In [7]:
# Remove spaces from column names - makes them easier to work with
transactions.columns = transactions.columns.str.replace(' ', '_')
discounts.columns = discounts.columns.str.replace(' ', '_')
products.columns = products.columns.str.replace(' ', '_')
stores.columns = stores.columns.str.replace(' ', '_')

print(transactions.columns.tolist())

['Invoice_ID', 'Line', 'Customer_ID', 'Product_ID', 'Size', 'Color', 'Unit_Price', 'Quantity', 'Date', 'Discount', 'Line_Total', 'Store_ID', 'Employee_ID', 'Currency', 'Currency_Symbol', 'SKU', 'Transaction_Type', 'Payment_Method', 'Invoice_Total']


In [8]:
# Fix typo in discounts column name
discounts = discounts.rename(columns={'Discont': 'Discount'})
print(discounts.columns.tolist())

['Start', 'End', 'Discount', 'Description', 'Category', 'Sub_Category']


Converting Date from text to datetime so we can do time-based analysis
like monthly sales trends.

In [9]:
# Fix Date column - currently stored as text, needs to be datetime
transactions['Date'] = pd.to_datetime(transactions['Date'])
print(transactions['Date'].dtype)
print(transactions['Date'].head(3))

datetime64[ns]
0   2023-01-01 15:42:00
1   2023-01-01 15:42:00
2   2023-01-01 15:42:00
Name: Date, dtype: datetime64[ns]


Dropping Color column — 68% of values are missing and color is not
relevant to our discount effectiveness analysis.

In [10]:
# Drop Color - too many missing values and not needed for our analysis
transactions = transactions.drop(columns=['Color'])
print("Columns remaining:", transactions.columns.tolist())

Columns remaining: ['Invoice_ID', 'Line', 'Customer_ID', 'Product_ID', 'Size', 'Unit_Price', 'Quantity', 'Date', 'Discount', 'Line_Total', 'Store_ID', 'Employee_ID', 'Currency', 'Currency_Symbol', 'SKU', 'Transaction_Type', 'Payment_Method', 'Invoice_Total']


In [11]:
print("Columns remaining:", transactions.columns.tolist())


Columns remaining: ['Invoice_ID', 'Line', 'Customer_ID', 'Product_ID', 'Size', 'Unit_Price', 'Quantity', 'Date', 'Discount', 'Line_Total', 'Store_ID', 'Employee_ID', 'Currency', 'Currency_Symbol', 'SKU', 'Transaction_Type', 'Payment_Method', 'Invoice_Total']


In [12]:
# Dropping columns not needed for discount analysis
transactions = transactions.drop(columns=['Currency_Symbol', 'SKU', 'Line', 'Employee_ID', 'Size'])
print("Columns remaining:", transactions.columns.tolist())

Columns remaining: ['Invoice_ID', 'Customer_ID', 'Product_ID', 'Unit_Price', 'Quantity', 'Date', 'Discount', 'Line_Total', 'Store_ID', 'Currency', 'Transaction_Type', 'Payment_Method', 'Invoice_Total']


In [13]:
print(transactions.isna().sum())

Invoice_ID          0
Customer_ID         0
Product_ID          0
Unit_Price          0
Quantity            0
Date                0
Discount            0
Line_Total          0
Store_ID            0
Currency            0
Transaction_Type    0
Payment_Method      0
Invoice_Total       0
dtype: int64


In [14]:
print(transactions[['Unit_Price', 'Quantity', 'Discount', 'Line_Total', 'Invoice_Total']].describe().round(2))

       Unit_Price   Quantity    Discount  Line_Total  Invoice_Total
count  6077200.00  6077200.0  6077200.00  6077200.00     6077200.00
mean       132.48        1.1        0.13      127.70         274.36
std        185.12        0.4        0.20      203.72         519.09
min          2.00        1.0        0.00        1.40           1.40
25%         32.50        1.0        0.00       27.45          39.00
50%         51.00        1.0        0.00       46.50          92.25
75%        116.50        1.0        0.25      120.75         256.55
max       1153.50        3.0        0.60     3460.50        8977.00


## 4. Feature Engineering

Creating new columns to support the discount analysis:
- Discount_Percentage: converting discount from decimal to percentage (easier to read)
- Discounted: a simple yes/no flag for whether a discount was applied
- Month and Year: extracted from Date for time-based trend analysis

In [15]:
# Convert discount from decimal to percentage
transactions['Discount_Percentage'] = (transactions['Discount'] * 100).round(0).astype(int)

# Flag whether a discount was applied
transactions['Discounted'] = transactions['Discount'] > 0

# Extract month and year for trend analysis
transactions['Year'] = transactions['Date'].dt.year
transactions['Month'] = transactions['Date'].dt.month

print(transactions[['Discount', 'Discount_Percentage', 'Discounted', 'Year', 'Month']].head(5))

   Discount  Discount_Percentage  Discounted  Year  Month
0       0.0                    0       False  2023      1
1       0.4                   40        True  2023      1
2       0.4                   40        True  2023      1
3       0.4                   40        True  2023      1
4       0.0                    0       False  2023      1


## 5. Cleaning Supporting Files

Cleaning discounts, products and stores before exporting everything
for SQL analysis.

In [16]:
# --- DISCOUNTS ---
# Convert Start and End from text to datetime
discounts['Start'] = pd.to_datetime(discounts['Start'])
discounts['End'] = pd.to_datetime(discounts['End'])

# Fill 10 missing Category and Sub_Category values with 'Unknown'
discounts['Category'] = discounts['Category'].fillna('Unknown')
discounts['Sub_Category'] = discounts['Sub_Category'].fillna('Unknown')

print("Discounts cleaned:")
print(discounts.dtypes)
print(discounts.isna().sum())

Discounts cleaned:
Start           datetime64[ns]
End             datetime64[ns]
Discount               float64
Description             object
Category                object
Sub_Category            object
dtype: object
Start           0
End             0
Discount        0
Description     0
Category        0
Sub_Category    0
dtype: int64


In [17]:
# --- PRODUCTS ---
# Only keep English description, drop other languages
# Drop Color and Sizes - not needed for discount analysis
products = products.drop(columns=[
    'Description_PT', 'Description_DE', 'Description_FR',
    'Description_ES', 'Description_ZH', 'Color', 'Sizes'
])

print("Products cleaned:")
print(products.columns.tolist())
print(products.isna().sum())

Products cleaned:
['Product_ID', 'Category', 'Sub_Category', 'Description_EN', 'Production_Cost']
Product_ID         0
Category           0
Sub_Category       0
Description_EN     0
Production_Cost    0
dtype: int64


## 6. Exporting Clean Data

Exporting all cleaned files for SQL analysis and Tableau visualisation.

In [18]:
# Export all cleaned files
transactions.to_csv('transactions_cleaned.csv', index=False)
discounts.to_csv('discounts_cleaned.csv', index=False)
products.to_csv('products_cleaned.csv', index=False)
stores.to_csv('stores_cleaned.csv', index=False)

files.download('transactions_cleaned.csv')
files.download('discounts_cleaned.csv')
files.download('products_cleaned.csv')
files.download('stores_cleaned.csv')

print("✅ All files exported!")
print("transactions:", transactions.shape)
print("discounts:", discounts.shape)
print("products:", products.shape)
print("stores:", stores.shape)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files exported!
transactions: (6077200, 17)
discounts: (181, 6)
products: (17940, 5)
stores: (35, 8)


## 7. SQL Analysis

Using sqlite3 to load the cleaned data into an in-memory database
and answer key business questions about discount effectiveness.

In [19]:
import pandas as pd
import numpy as np
import io
from google.colab import files
uploaded = files.upload()

Saving discounts_cleaned.csv to discounts_cleaned (1).csv
Saving products_cleaned.csv to products_cleaned (1).csv
Saving stores_cleaned.csv to stores_cleaned (1).csv
Saving transactions_cleaned.csv to transactions_cleaned (1).csv


In [27]:
print(uploaded.keys())

dict_keys(['discounts_cleaned (1).csv', 'products_cleaned (1).csv', 'stores_cleaned (1).csv', 'transactions_cleaned (1).csv'])


In [28]:
trans = pd.read_csv(io.BytesIO(uploaded['transactions_cleaned (1).csv']))
disc = pd.read_csv(io.BytesIO(uploaded['discounts_cleaned (1).csv']))
prod = pd.read_csv(io.BytesIO(uploaded['products_cleaned (1).csv']))
stor = pd.read_csv(io.BytesIO(uploaded['stores_cleaned (1).csv']))

print("trans:", trans.shape)
print("disc:", disc.shape)
print("prod:", prod.shape)
print("stor:", stor.shape)

trans: (6077200, 17)
disc: (181, 6)
prod: (17940, 5)
stor: (35, 8)


In [29]:
import sqlite3

conn = sqlite3.connect(':memory:')

trans.to_sql('transactions', conn, if_exists='replace', index=False)
disc.to_sql('discounts', conn, if_exists='replace', index=False)
prod.to_sql('products', conn, if_exists='replace', index=False)
stor.to_sql('stores', conn, if_exists='replace', index=False)

print("✅ Database ready!")

✅ Database ready!


In [30]:
stor['Country'] = stor['Country'].str.replace('中国', 'China')
stor.to_sql('stores', conn, if_exists='replace', index=False)

35

### Q1: What percentage of transactions were discounted vs full price?

In [31]:
q1 = pd.read_sql("""
    SELECT
        Discounted,
        COUNT(*) as Total_Transactions,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as Percentage
    FROM transactions
    GROUP BY Discounted
""", conn)

print(q1)

   Discounted  Total_Transactions  Percentage
0           0             4288334       70.56
1           1             1788866       29.44


**Finding:** 70.6% of transactions were at full price and 29.4% were discounted.
This means nearly 1 in 3 purchases involved a discount, already suggesting
discounting is a significant part of this retailer's strategy.

### Q2: Do discounted transactions have higher quantities than non-discounted?

In [32]:
q2 = pd.read_sql("""
    SELECT
        Discounted,
        ROUND(AVG(Quantity), 2) as Avg_Quantity,
        ROUND(AVG(Line_Total), 2) as Avg_Sale_Value
    FROM transactions
    GROUP BY Discounted
""", conn)

print(q2)

   Discounted  Avg_Quantity  Avg_Sale_Value
0           0           1.1          147.35
1           1           1.1           80.59


**Finding:** Discounted transactions generate almost half the revenue per
transaction (80.59) compared to full price transactions (147.35), yet
the average quantity purchased is identical at 1.1 items in both cases.

This suggests discounts are not driving customers to buy more,they are
simply reducing revenue per sale. The customer was likely going to buy anyway.

### Q3: Which discount percentage drives the highest average sale value?

In [33]:
q3 = pd.read_sql("""
    SELECT
        Discount_Percentage,
        COUNT(*) as Total_Transactions,
        ROUND(AVG(Line_Total), 2) as Avg_Sale_Value
    FROM transactions
    WHERE Discount_Percentage > 0
    GROUP BY Discount_Percentage
    ORDER BY Avg_Sale_Value DESC
""", conn)

print(q3)

   Discount_Percentage  Total_Transactions  Avg_Sale_Value
0                   20              133528          117.29
1                   25              147943           92.63
2                   35              220981           82.50
3                   45              230070           78.78
4                   50              899045           75.30
5                   40              122473           69.82
6                   60               34826           62.75


**Finding:** Higher discounts do not translate to higher revenue per transaction.
In fact, the 20% discount generates the highest average sale value (117.29)
among all discount levels, while the 60% discount generates the lowest (62.75).

However, the 50% discount drives the highest transaction volume (899,045 transactions),
suggesting customers respond to larger discounts in terms of purchase frequency
but not in terms of spending more per visit.

This raises a key business question: is the 50% discount worth it if it
generates less revenue per sale than a 20% discount?

### Q4: Which product categories get discounted the most?

In [34]:
q4 = pd.read_sql("""
    SELECT
        p.Category,
        COUNT(*) as Total_Transactions,
        ROUND(AVG(t.Discount_Percentage), 2) as Avg_Discount,
        ROUND(AVG(t.Line_Total), 2) as Avg_Sale_Value
    FROM transactions t
    JOIN products p ON t.Product_ID = p.Product_ID
    WHERE t.Discounted = 1
    GROUP BY p.Category
    ORDER BY Avg_Discount DESC
""", conn)

print(q4)

    Category  Total_Transactions  Avg_Discount  Avg_Sale_Value
0   Feminine              784934         43.91           78.20
1  Masculine              675062         43.23           87.26
2   Children              328870         38.76           72.57


**Finding:** Feminine products receive the highest average discount (43.91%)
and account for the most discounted transactions (784,934), followed by
Masculine (43.23%) and Children (38.76%).

However, higher discounting does not translate to higher sale value.
Masculine products generate the highest average sale value (87.26) despite
receiving a lower average discount than Feminine products (78.20).

Children's products receive the least discounting yet also generate the
lowest average sale value (72.57), suggesting discounts alone do not
drive revenue in this category.

### Q5: Which months have the highest discount rates?

Looking at how discount rates and sale values change across months and years
to identify any seasonal patterns in discounting behaviour.

In [35]:
q5 = pd.read_sql("""
    SELECT
        Month,
        Year,
        ROUND(AVG(Discount_Percentage), 2) as Avg_Discount,
        COUNT(*) as Total_Transactions,
        ROUND(AVG(Line_Total), 2) as Avg_Sale_Value
    FROM transactions
    GROUP BY Year, Month
    ORDER BY Year, Month
""", conn)
print(q5)

    Month  Year  Avg_Discount  Total_Transactions  Avg_Sale_Value
0       1  2023         10.36              162503           91.04
1       2  2023          0.00              112879          106.70
2       3  2023         12.20              249683          128.38
3       4  2023          0.00              177150          139.73
4       5  2023          8.82              201920          125.93
5       6  2023          0.00              125348          156.08
6       7  2023          0.00              139866          130.26
7       8  2023          0.00              121703          140.50
8       9  2023         16.01              293148          123.24
9      10  2023          5.05              259879          146.16
10     11  2023          5.09              186271          133.29
11     12  2023         33.86              630401           95.38
12      1  2024          9.52              169036          106.88
13      2  2024          0.00              136876          124.19
14      3 

**Finding:** December consistently shows the highest discount rates
(33.86% in 2023, 34.09% in 2024) and the highest transaction volumes
(630,401 and 692,348 respectively), suggesting heavy promotional
activity during the holiday season.

However, months with zero discounting (February, April, June, July, August)
consistently show higher average sale values than heavily discounted months.
For example, June 2024 achieved an average sale value of 169.89 with no
discount, compared to December 2024's 108.25 with a 34% average discount.

This pattern reinforces the finding from Q2, customers spend more per
transaction at full price than when discounts are applied, even during
high-traffic periods like December.

### Q6: Which countries generate the most revenue, discounted vs full price?

Comparing total and average revenue across countries, split by whether
a discount was applied or not.

In [36]:
q6 = pd.read_sql("""
    SELECT
        s.Country,
        t.Discounted,
        COUNT(*) as Total_Transactions,
        ROUND(SUM(t.Line_Total), 2) as Total_Revenue,
        ROUND(AVG(t.Line_Total), 2) as Avg_Sale_Value
    FROM transactions t
    JOIN stores s ON t.Store_ID = s.Store_ID
    GROUP BY s.Country, t.Discounted
    ORDER BY s.Country, t.Discounted
""", conn)
print(q6)

           Country  Discounted  Total_Transactions  Total_Revenue  \
0            China           0             1050219   4.652907e+08   
1            China           1              421297   1.050168e+08   
2      Deutschland           0              488778   2.525333e+07   
3      Deutschland           1              221573   6.362154e+06   
4           España           0              381291   1.966649e+07   
5           España           1              159251   4.584769e+06   
6           France           0              430732   2.210646e+07   
7           France           1              183393   5.272193e+06   
8         Portugal           0              386653   1.993636e+07   
9         Portugal           1              162448   4.673117e+06   
10  United Kingdom           0              411301   1.483799e+07   
11  United Kingdom           1              181801   3.596036e+06   
12   United States           0             1139360   6.481341e+07   
13   United States           1    

**Finding:** China generates by far the highest average sale value per
transaction, 443.04 for full price and 249.27 for discounted transactions.
This is likely due to currency differences (CNY vs USD/EUR) rather than
genuinely higher spending, and should be normalised in future analysis.

Across all countries, the pattern is completely consistent, full price
transactions generate significantly higher average sale values than
discounted ones. For example in the US, full price averages 56.89 vs
31.91 discounted. In France, 51.32 vs 28.75.

The United Kingdom shows the lowest average sale values overall
(36.08 full price, 19.78 discounted), suggesting either a different
product mix or pricing strategy compared to other markets.

This consistent pattern across all 7 countries strongly supports the
core finding, discounting reduces revenue per transaction everywhere,
not just in specific markets.

### Q7: Average revenue per transaction by category — discounted vs full price

Breaking down sale value by product category to see which categories
are most impacted by discounting.

In [37]:
q7 = pd.read_sql("""
    SELECT
        t.Discounted,
        p.Category,
        ROUND(AVG(t.Line_Total), 2) as Avg_Sale_Value,
        ROUND(AVG(t.Discount_Percentage), 2) as Avg_Discount,
        COUNT(*) as Total_Transactions
    FROM transactions t
    JOIN products p ON t.Product_ID = p.Product_ID
    GROUP BY t.Discounted, p.Category
    ORDER BY p.Category, t.Discounted
""", conn)
print(q7)

   Discounted   Category  Avg_Sale_Value  Avg_Discount  Total_Transactions
0           0   Children          106.76          0.00              421815
1           1   Children           72.57         38.76              328870
2           0   Feminine          140.07          0.00             2138428
3           1   Feminine           78.20         43.91              784934
4           0  Masculine          166.28          0.00             1728091
5           1  Masculine           87.26         43.23              675062


**Finding:** Across all three product categories, discounting consistently
reduces average sale value by almost half.

Masculine products show the biggest drop — from 166.28 at full price to
87.26 with a discount (47% reduction). Feminine products drop from 140.07
to 78.20 (44% reduction), and Children from 106.76 to 72.57 (32% reduction).

Children's products are least impacted by discounting in terms of revenue
drop, yet still account for the lowest sale values in both discounted and
full price transactions.

This consistent pattern across all categories further confirms that
discounting is not driving higher spending — it is simply

In [38]:
q1.to_csv('q1_discount_split.csv', index=False)
q2.to_csv('q2_quantity_analysis.csv', index=False)
q3.to_csv('q3_discount_effectiveness.csv', index=False)
q4.to_csv('q4_category_discounts.csv', index=False)
q5.to_csv('q5_monthly_trends.csv', index=False)
q6.to_csv('q6_country_revenue.csv', index=False)
q7.to_csv('q7_category_breakdown.csv', index=False)

files.download('q1_discount_split.csv')
files.download('q2_quantity_analysis.csv')
files.download('q3_discount_effectiveness.csv')
files.download('q4_category_discounts.csv')
files.download('q5_monthly_trends.csv')
files.download('q6_country_revenue.csv')
files.download('q7_category_breakdown.csv')

print("✅ All query results exported!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All query results exported!


In [39]:
# Save to a proper SQLite database file for DBeaver
conn_file = sqlite3.connect('fashion_retail.db')

trans.to_sql('transactions', conn_file, if_exists='replace', index=False)
disc.to_sql('discounts', conn_file, if_exists='replace', index=False)
prod.to_sql('products', conn_file, if_exists='replace', index=False)
stor.to_sql('stores', conn_file, if_exists='replace', index=False)

conn_file.close()
files.download('fashion_retail.db')
print("✅ Database file exported!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Database file exported!
